### Notebook 02: Data Preparation & Modeling
Metadatos:
Autor: Eric Acosta
19/04/2026
Version 1.0

**Agentic Retention Platform (ARP)**
**Project Context**
Following the CRISP-DM framework, we have completed the Data Understanding (Phase 2). We are now entering Data Preparation (Phase 3) and Modeling (Phase 4). Our goal is to transform raw telemetry into a high-performance feature set for an XGBoost model, which will then serve as the "brain" for our Agentic Retention Platform.

**Phase 2 Findings: The Blueprint (Source of Truth)**
**Note:** These findings were extracted from the initial audit and guide our current pipeline.
Target = 'Churn Label'

**1. Data Purge: The "Dead Weight" & Leakage Control**
**Cardinality = 1:** Remove Quarter, Country, State.
**Identifiers:** Drop Customer ID from training (retain as metadata).
**Geographic Noise:** Zip Code is redundant.
**Data Leakage:** Remove Customer Status, Churn Score, Churn Category, and Churn Reason.
**Financial Redundancy:** Drop Total Revenue; prioritize Total Charges.

**2. Logic & Consistency: The "Guardrails"**
**Binary Target:** Map Churn Label to 1 (Yes) and 0 (No).
**The Ghost Filter:** Exclude Tenure == 0 for training (to be managed separately by the Platform).
**Service Dependency:** If Internet Service == 'No', then all internet add-ons (Tech Support, etc.) must be 'No internet service'.
**Financial Drift:** Create a flag for records where (Monthly Charges * Tenure) deviates >15% from Total Charges.

**3. Feature Transformation**
**Contextual Imputation:** Offer -> 'No Offer'; Internet Type -> 'No Internet'.
**Encoding:** One-Hot Encoding for low-cardinality categorical variables.

**4. Feature Engineering**
**Price Shock:** Interaction between Contract Type and Monthly Charge.
**Loyalty Curve:** Analyze the full 72-month Tenure.
**Cluster Regions:** Apply K-Means to Latitude and Longitude to create 5 geographic risk zones.

**5. Modeling Strategy**
**Cost-Sensitive Learning:** Use scale_pos_weight in XGBoost to penalize False Negatives (Churners missed).
**The Vault (Surgical Split):** Isolate 10% of RAW data immediately to simulate the Agent's real-world environment.

### PHASE 3 Data preparation
**Output**
**Dataset**
These are the dataset(s) produced by the data preparation phase, used for modeling or for the major
analysis work of the project.

**Output**
**Dataset description**
This is the description of the dataset(s) used for the modeling or for the major analysis work of the project.

**3.1 Select data**
**Task Select data**
Decide on the data to be used for analysis. Criteria include relevance to the data mining goals, quality,
and technical constraints such as limits on data volume or data types.
**Output**
**Rationale for inclusion/exclusion**
List the data to be used/excluded and the reasons for these decisions.
Activities
* Collect appropriate additional data (from different sources—in-house as well as externally)
* Perform significance and correlation tests to decide if fields should be included
* Reconsider Data Selection Criteria (See Task 2.1) in light of experiences of data quality and data
exploration (i.e., may wish include/exclude other sets of data)
* Reconsider Data Selection Criteria (See Task 2.1) in light of experience of modeling (i.e., model
assessment may show that other datasets are needed)
* Select different data subsets (e.g., different attributes, only data which meet certain conditions)
* Consider the use of sampling techniques (e.g., A quick solution may involve splitting test and training
datasets or reducing the size of the test dataset, if the tool cannot handle the full dataset. It may also
be useful to have weighted samples to give different importance to different attributes or different
values of the same attribute.)
* Document the rationale for inclusion/exclusion
* Check available techniques for sampling data
**Good idea!** Based on Data Selection Criteria, decide if one or more attributes are more important than others and weight the attributes accordingly. Decide, based on the context (i.e., application, tool, etc.), how to handle the weighting.

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import warnings
import logging
from typing import Optional
from sklearn.model_selection import train_test_split
from IPython.display import display, HTML

# Logger Configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s - %(message)s",
    force=True
)

#Display settings
%matplotlib inline
sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')

In [15]:
# Definition of route and procedural load

def ingest_and_isolate_vault(
    file_path: str,
    vault_output_path: str,
    test_size: float=0.10,
    random_state: int=42
    ) -> Optional[pd.DataFrame]:
    """
    Reads raw data from a compressed CSV, splits it using stratified sampling
    based on 'Churn Label' and securely save the holdout set.
    
    Args:
        file_path (str): Path to the raw ZIP compressed CSV file.
        vault_output_path (str): File path where the holdout set (The Vault) will be saved. 
        test_file (float, optional): Proportion of the dataset to include in the holdout split. Default to 0.10
        random_state (int, optional): Controls the  shuffling applied to the data before splitting.
        
    Returns:
        Optional[pd.DataFrame]: The develoment DataFrame (90%) for processing and cleaning for training.
        Returns None if critical extraction occurs.
        
    Raises:
        FileNotFoundError: If the specified file_path does not exist
        pd.errors.EmptyDataError: If the CSV file is empty.
        KeyError: If 'Churn Label' is missing from the dataset.
    """
    try:
        logging.info(f"Starting ingestion from: {file_path}")
        # 1. Procedural Load
        df_raw = pd.read_csv(file_path, compression='zip')
        
        # 2. Stratified Split
        # We target 'Churn Label' to maintain class balance in both sets
        df_dev, df_vault = train_test_split(
            df_raw,
            test_size=test_size,
            random_state=random_state,
            stratify=df_raw['Churn Label']
        )
        
        # 3. The base directory is created if it does not exist, based on the provided path.
        os.makedirs(os.path.dirname(vault_output_path), exist_ok=True)
        df_vault.to_csv(vault_output_path, index=False)
        
        logging.info(f"The Vault ({(test_size * 100):.0f}%) locked at: {vault_output_path}")
        logging.info(f"Development Set ({(1 - test_size) * 100:.0f}%) ready: {df_dev.shape[0]} rows.")
        
        return df_dev
    
    except (FileNotFoundError, pd.errors.EmptyDataError, KeyError) as e:
        logging.error(f"Data validation error during ingestion: {e}")
        return None

# --- Execution ---
DATA_PATH = "../data/raw/telco.zip"
VAULT_PATH = "../data/processed/holdout_life_simulation.csv"

df_dev = ingest_and_isolate_vault(
    file_path=DATA_PATH,
    vault_output_path=VAULT_PATH
    )
    

2026-04-24 23:03:34,854 INFO - Starting ingestion from: ../data/raw/telco.zip
2026-04-24 23:03:34,959 INFO - The Vault (10%) locked at: ../data/processed/holdout_life_simulation.csv
2026-04-24 23:03:34,959 INFO - Development Set (90%) ready: 6338 rows.


### Structural removal
Drop the columns identified on phase 2(Leakage, Zero-variance, Redundant)
Remove: 'Quarter', 'Country', 'State', 'Zip Code', 'Customer Status', 'Churn Score', 'Churn Category', 'Churn Reason', 'Total Revenue'

In [ ]:
def structural_removal(
    columns_to_drop: list,
    df: pd.DataFrame
    ) -> Optional [pd.DataFrame]:
    """
    Removes a specified list of columns from a DataFrame.
    
    Args:
        columns_to_drop (list): List of column names to be dropped.
        df (pd.DataFrame): The input DataFrame.
        
    Returns:
        Optional[pd.DataFrame]: The DataFrame without the specified columns.
                                Returns None if the dropping process fails.
    Raises:
        KeyError: If one or more columns in the list are not found in the DataFrame.
    """
    # Preventing DF None
    if df is None:
        logging.error("Critical: Received a  NoneType object instead of a DataFrame. Pipeline aborted")
        return None
    
    try:
        # Inmutability: Work on a copy
        # Execution
        df_out = df.drop(columns=columns_to_drop)
        
        logging.info(f"Successfully dropped {len(columns_to_drop)} columns: {columns_to_drop}")
        logging.info(f"Quantity of Columns before dropping: {df.shape[1]} and after: {df_out.shape[1]}")
        return df_out
     
    except KeyError as e:
        logging.error(f"Failed to drop Columns. Column not found {e}")
        return None
    except Exception as e:
        logging.error(f"Unexpected error during structural removal: {e}")
        return None


# --- Execution ---
cols_to_drop = ['Quarter', 'Country', 'State', 'Zip Code', 'Customer Status', 'Churn Score', 'Churn Category', 'Churn Reason', 'Total Revenue']

df_dev = structural_removal(cols_to_drop, df_dev)


    

2026-04-24 23:03:38,045 INFO - Successfully dropped 9 columns: ['Quarter', 'Country', 'State', 'Zip Code', 'Customer Status', 'Churn Score', 'Churn Category', 'Churn Reason', 'Total Revenue']
2026-04-24 23:03:38,047 INFO - Quantity of Columns before dropping: 50 and after: 41


### Metadata Standardization (Column Naming)
Working with inconsistent column names is a nightmare that causes constant KeyErrors.
**The Action:** Convert all column names to lowercase, replace spaces with underscores (_), and remove special characters (parentheses, percentages, etc.) to enforce snake_case.
**The Risk Avoided:** Prevents syntax errors and ensures your code complies with Python's PEP8 formatting standards.

In [ ]:
def standarize_columns(
    df: pd.DataFrame
    ) -> Optional[pd.DataFrame]:
    """
    Standardizes column names to snake_case.
    
    Args:
        df(pd.DataFrame): The input DataFrame
        
    Returns:
        df(pd.DataFrame): The DataFrame with the Standarize Columns
                          Returns None if the cleaning process fails
                          
    Raise:
        Cath all unexpected Memory or Pandas Error
    """
    
    # Defensive Check
    if df is None:
        logging.error(f"Critical: Received NoneType instead of DataFrame")
        return None
    
    try:
        # Inmutability: Work on a copy
        df_out = df.copy()
        
        # Execution()
        df.columns = (
            df.columns.str.strip().str.lower()
            .str.replace(r'[^\w\s]', '', regex=True)
            .str.replace(r'\s+', '_', regex=True)
        )
        logging.info("Column names succesfully standarized to snake_case")
        return df_out
    
    except Exception as e:
        # Catch-all for unexpected memory or pandas errors
        logging.error(f"Unexpected error during column standarization {e}")
        return None
    
    
# --- Execution ---
df_dev = standarize_columns(df_dev)

2026-04-24 22:54:21,587 INFO - Column names succesfully standarized to snake_case
